This is a public presentation copy: code and final explanations are retained; learning drafts remain in the original local notebook. Saved outputs and local metadata are cleared. Run from the notebooks directory after installing the project and obtaining the source CSV. See ../reports/model_card.md for the final experiment results and ../docs/bank_transfer_design.md for the separate hypothetical bank requirements. Ablation and interaction ideas in EDA are proposed checks, not completed experiments.


# Stage 3 — Leakage-Safe Baseline Modeling

## Objective and boundary

This stage builds a no-skill DummyClassifier and an interpretable Logistic Regression baseline under the same five stratified cross-validation folds. The 20% holdout is created once and then sealed: it may appear in split-integrity checks, but it must not be used to fit preprocessing, train a model, compare models, tune hyperparameters, or choose a threshold during Stage 3.



In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, precision_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    average_precision_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
)

In [ ]:
RANDOM_STATE = 42

In [ ]:
data_path = Path("../data/raw/customer_churn.csv")
assert data_path.exists(),f"Dataset not found: {data_path.resolve()}"

In [ ]:
df_raw = pd.read_csv(data_path)
df = df_raw.copy()

assert df.shape == (7043, 21)
assert df["customerID"].is_unique
assert set(df["Churn"].unique()) == {"No", "Yes"}

In [ ]:
target = df["Churn"].map({"No": 0, "Yes": 1})
customer_ids = df["customerID"].copy()
features = df.drop(columns=["customerID","Churn"]).copy()

In [ ]:
assert target.notna().all()
assert set(target.unique()) == {0,1}
assert len(features.columns) ==19
assert "customerID" not in features.columns
assert "Churn" not in features.columns

In [ ]:
eda_only_columns = ["ChurnFlag","TenureBand","RiskCombo"]

In [ ]:
for column in eda_only_columns:
    assert column not in features.columns

In [ ]:
features["TotalCharges"] = pd.to_numeric(
    features["TotalCharges"],
    errors="coerce",
)

In [ ]:
features["SeniorCitizen"] = features["SeniorCitizen"].astype(str)
assert features["TotalCharges"].isna().sum() == 11

In [ ]:
(
X_train,
X_holdout,
y_train,
y_holdout,
id_train,
id_holdout,
) = train_test_split(
    features,
    target,
    customer_ids,
    test_size=0.20,
    stratify = target,
    random_state=RANDOM_STATE,
)

In [ ]:
assert len(X_train) == 5634
assert len(X_holdout) == 1409
assert X_train.index.equals(y_train.index)
assert X_holdout.index.equals(y_holdout.index)
assert set(id_train).isdisjoint(set(id_holdout))
assert abs(y_train.mean() - target.mean()) < 0.005
assert abs(y_holdout.mean() - target.mean()) < 0.005

In [ ]:
split_audit = pd.DataFrame(
    {
        "rows":[len(y_train),len(y_holdout)],
        "churn_count": [int(y_train.sum()),int(y_holdout.sum())],
        "churn_rate": [y_train.mean(), y_holdout.mean()],
    },
    index=["training","holdout"],
)

split_audit

In [ ]:
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]

In [ ]:
categorical_features = [
    column
    for column in X_train.columns
    if column not in numeric_features
]

In [ ]:
assert len(numeric_features) == 3
assert len(categorical_features) == 16
assert set(numeric_features + categorical_features) == set(X_train.columns)

In [ ]:
numeric_pipline = Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
    ]
)

In [ ]:
categorical_pipline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True,
            ),
        ),
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric",numeric_pipline,numeric_features),
        ("categorical",categorical_pipline,categorical_features),
    ],
    remainder="drop"
)

In [ ]:
# Original learning attempt, preserved for the audit trail:
# preprocessor.fit(features)
# preprocessor.fit(X_train)
#
# These standalone fits are intentionally not executed. Fitting on `features`
# would use holdout statistics, while fitting once on all of `X_train` before
# cross-validation would expose each validation fold to preprocessing statistics.
# The cloned preprocessor inside each model Pipeline is fitted separately on the
# training portion of every cross-validation fold.
preprocessor

In [ ]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocess",clone(preprocessor)),
        ("model",DummyClassifier(strategy="prior")),
    ]
)

In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        ("preprocess", clone(preprocessor)),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                solver="lbfgs",
            ),
        ),
    ]
)

In [ ]:
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state=RANDOM_STATE,
)

In [ ]:
cv_splits = list(cv.split(X_train,y_train))
assert len(cv_splits) == 5

In [ ]:
for fold_train_idx, fold_valid_idx in cv_splits:
    assert set(fold_train_idx).isdisjoint(set(fold_valid_idx))

In [ ]:
scoring = {
    "ap": "average_precision",
    "recall": "recall",
    "precision": make_scorer(precision_score, zero_division=0),
    "f1": "f1",
    "roc_auc":"roc_auc",
    "accuracy": "accuracy",
}

In [ ]:
print(scoring)

In [ ]:
models = {
    "Dummy" : dummy_pipeline,
    "Logistic Regression": logistic_pipeline,
}
comparison_rows = []
for model_name, model_pipeline in models.items():
    cv_scores = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=cv_splits,
        scoring=scoring,
        return_train_score=False,
        n_jobs = 1,
        error_score = "raise",
    )
    row = {"model":model_name}

    for metric_name in scoring:
        fold_values = cv_scores[f"test_{metric_name}"]
        row[f"{metric_name}_mean"] = fold_values.mean()
        row[f"{metric_name}_std"] = fold_values.std(ddof=1)
    comparison_rows.append(row)
baseline_cv_results = (
    pd.DataFrame(comparison_rows)
    .set_index("model")
    .sort_values("ap_mean",ascending=False)
)
baseline_cv_results


expected_metrics = set(scoring)

assert set(baseline_cv_results.index) == {
    "Dummy",
    "Logistic Regression",
}

for metric_name in expected_metrics:
    assert baseline_cv_results[f"{metric_name}_mean"].between(0, 1).all()
    assert baseline_cv_results[f"{metric_name}_std"].ge(0).all()


In [ ]:
print(baseline_cv_results)

## My Stage 3 Baseline Interpretation

### 1. Dummy baseline

- **Numerical evidence:** The DummyClassifier achieved an Average Precision of 0.265 ± 0.000, a ROC-AUC of 0.500 ± 0.000, and an Accuracy of 0.735 ± 0.000. Its Recall, Precision, and F1 score for the churn class were all 0.000.

- **What the model is doing:** The DummyClassifier uses the class prior and predicts the majority non-churn class for every customer at the default decision rule. It does not learn meaningful relationships between customer features and churn.

- **Why Accuracy alone is misleading:** Approximately 73.5% of the training customers did not churn, so the DummyClassifier achieved about 73.5% Accuracy while failing to identify any churned customers. Its churn Recall of 0 demonstrates why Accuracy alone is insufficient for this imbalanced classification problem.

### 2. Logistic Regression baseline

- **Numerical evidence compared with Dummy:** Logistic Regression achieved an Average Precision of 0.661 ± 0.022, compared with 0.265 ± 0.000 for the DummyClassifier. Its ROC-AUC increased from 0.500 to 0.846 ± 0.014, and its Accuracy increased from 0.735 to 0.803 ± 0.013. At the default classification threshold, Logistic Regression achieved a Recall of 0.543 ± 0.046, a Precision of 0.655 ± 0.031, and an F1 score of 0.593 ± 0.033, while all three metrics were 0 for the DummyClassifier.

- **Which metric improved most:** Precision showed the largest absolute numerical increase, from 0 to approximately 0.655, while Recall increased from 0 to approximately 0.543. However, improvements across different metrics should not be compared mechanically because they answer different questions. Average Precision is the primary ranking metric in this project, and its increase from 0.265 to 0.661 provides strong evidence that Logistic Regression ranks churned customers more effectively than the no-skill baseline.

- **Whether five-fold variability appears acceptable:** The cross-validation variability appears reasonably small. For example, the standard deviation was approximately 0.022 for Average Precision and 0.014 for ROC-AUC. Recall showed somewhat greater variability at approximately 0.046, but the Logistic Regression model still outperformed the DummyClassifier consistently across the training folds. This stability applies only to the current stratified cross-validation design and does not prove stability on future customer populations.

### 3. Current decision

- **What Logistic Regression has established:** Logistic Regression has established that the feature matrix contains predictive information beyond the overall churn prevalence and the majority-class strategy. It provides a meaningful and interpretable baseline for customer-level churn-risk ranking.

- **What has not yet been established:** Logistic Regression has not been established as the final model. The final holdout has not been evaluated, the business classification threshold has not been selected, and the current results do not demonstrate causal relationships or verified retention impact. At the default threshold, the model identifies approximately 54.3% of churned customers, so the trade-off between missed churners, false positives, and outreach capacity still needs to be evaluated.

- **Why XGBoost is still worth testing in Stage 4:** Logistic Regression primarily models linear relationships in the log-odds unless interactions are explicitly engineered. XGBoost is worth testing because it may capture nonlinear patterns and feature interactions. It should be evaluated using the same training folds and metrics, and its additional complexity should be accepted only if it produces stable and meaningful improvement over Logistic Regression.



From observations,logistic Regression own high and stable AP, and recall, it establishs that logistic models pass the Dummy rules, which is better than the simplist strategy. also accuracy, though cant serve as a evaluation index, demonstrate the features matrix includes predictive informations. Overall, if is effective in the cross0- in training datas


Based on five-fold cross-validation within the training data, Logistic Regression clearly outperformed the DummyClassifier. Its Average Precision increased from approximately 0.265 to 0.661, showing that the feature matrix contains meaningful predictive information beyond the majority-class strategy. At the default threshold, the model achieved approximately 54.3% Recall and 65.5% Precision. The fold-to-fold variability was relatively small, although Recall was less stable than AP and ROC-AUC. These results establish Logistic Regression as a useful baseline, but not necessarily the final model.

Leakage Audit

| Component | Data allowed to fit it | Current protection |
|---|---|---|
| TotalCharges type conversion | No fitted statistics | Deterministic conversion only |
| Numeric imputation | Current CV training fold | Inside Pipeline |
| StandardScaler | Current CV training fold | Inside Pipeline |
| One-Hot category vocabulary | Current CV training fold | Inside Pipeline |
| Logistic Regression parameters | Current CV training fold | Inside Pipeline |
| Model comparison | Training CV results only | Shared five folds |
| Threshold selection | Not performed in Stage 3 | Deferred |
| Final holdout | No model access yet | Sealed |

## Hypothetical transfer to a bank data-request workflow

This notebook uses the public IBM Telco dataset and is not a model built with bank data or deployed during my internship. Its transferable value is the workflow: before querying data or training a model, I would translate the business request into an observation unit, eligible population, prediction cutoff, observation and outcome windows, exclusions, target definition, and required data grain.

In a bank setting, exclusions would need to distinguish voluntary attrition from events such as confirmed fraud, involuntary or administrative closure, death, duplicate or test accounts, and system-driven closure. Field definitions and joins would be confirmed with the requesting department and data owner before SQL extraction. Access would follow authorization, masking, and the minimum-necessary principle: retrieve only the fields and customer population required by the approved work order, avoid exposing direct identifiers in modeling data, and document the extraction and validation checks.

A future risk score would be treated as a candidate list for business review and retention planning, not proof that every selected customer will leave or should automatically receive an intervention.

In [ ]:
xgb_numeric_pipline =Pipeline(
    steps=[
        ("imputer",SimpleImputer(strategy = "median")),
    ]
)
xgb_preprocessor =ColumnTransformer(
    transformers=[
        ("numeric",xgb_numeric_pipline,numeric_features),
        (
            "categorical",
            clone(categorical_pipline),
            categorical_features,
        ),
    ],
    remainder="drop",
)

In [ ]:
fixed_xgb_params = {
    "n_estimators":300,
    "learning_rate":0.05,
    "max_depth":3,
    "min_child_weight":1,
    "subsample":0.80,
    "colsample_bytree":0.80,
    "reg_lambda":1.0,
    "gamma":0.0,
}

xgb_fixed_pipeline = Pipeline(
    steps=[
        ("preprocess", clone(xgb_preprocessor)),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=1,
                verbosity=0,
                **fixed_xgb_params,
            ),
        ),
    ]
)

In [ ]:

def evaluate_cv_pipeline(model_name, model_pipeline):
    cv_scores = cross_validate(
        estimator=model_pipeline,
        X=X_train,
        y=y_train,
        cv=cv_splits,
        scoring=scoring,
        return_train_score=False,
        n_jobs=1,
        error_score="raise",
    )

    result_row = {"model": model_name}
    for metric_name in scoring:
        fold_values = cv_scores[f"test_{metric_name}"]
        result_row[f"{metric_name}_mean"] = fold_values.mean()
        result_row[f"{metric_name}_std"] = fold_values.std(ddof=1)

    return result_row, cv_scores

In [ ]:


fixed_xgb_row, fixed_xgb_cv_scores = evaluate_cv_pipeline(
    model_name = 'XGBoost fixed',
    model_pipeline= xgb_fixed_pipeline,
)
stage4_initial_comparison = pd.concat(
    [
        baseline_cv_results,
        pd.DataFrame([fixed_xgb_row]).set_index("model"),
    ]
).sort_values("ap_mean",ascending=False)

stage4_initial_comparison

In [ ]:
assert set(stage4_initial_comparison.index) == {
    "Dummy",
    "Logistic Regression",
    "XGBoost fixed",
}
assert stage4_initial_comparison.filter(like = "mean").notna().all().all()
assert stage4_initial_comparison.filter(like = "_std").ge(0).all().all()


In [ ]:
xgb_candidate_configs = [
    {
        "candidate": "A_base",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
        "gamma": 0.0,
    },
    {
        "candidate": "B_shallow",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 2,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
        "gamma": 0.0,
    },
    {
        "candidate": "C_deeper",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 4,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
        "gamma": 0.0,
    },
    {
        "candidate": "D_slow_learning",
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
        "gamma": 0.0,
    },
    {
        "candidate": "E_fast_learning",
        "n_estimators": 200,
        "learning_rate": 0.10,
        "max_depth": 3,
        "min_child_weight": 1,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 1.0,
        "gamma": 0.0,
    },
    {
        "candidate": "F_regularized",
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 3,
        "min_child_weight": 5,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_lambda": 5.0,
        "gamma": 0.10,
    },
]

In [ ]:
assert len(xgb_candidate_configs) == 6
assert len({config["candidate"] for config in xgb_candidate_configs}) == 6

In [ ]:
xgb_candidate_rows = []
xgb_candidate_cv_scores = {}
xgb_candidate_pipeline = {}


In [ ]:
for config in xgb_candidate_configs:
    candidate_name = config["candidate"]
    model_params = {
        key: value
        for key, value in config.items()
        if key != "candidate"
    }
    candidate_pipeline = Pipeline(
        steps=[
            ("preprocess", clone(xgb_preprocessor)),
            (
                "model",
                     XGBClassifier(
                        objective="binary:logistic",
                        eval_metric= "logloss",
                        tree_method="hist",
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                        verbosity=0,
                        **model_params,
                ),
            ),
        ]
    )
    result_row, cv_scores = evaluate_cv_pipeline(
        model_name=candidate_name,
        model_pipeline=candidate_pipeline,
    )
    xgb_candidate_rows.append(result_row)
    xgb_candidate_cv_scores[candidate_name] = cv_scores
    xgb_candidate_pipeline[candidate_name] = candidate_pipeline

xgb_search_results = (
    pd.DataFrame(xgb_candidate_rows)
    .set_index("model")
    .sort_values("ap_mean", ascending=False)
)

    
            
            

In [ ]:
xgb_search_results

In [ ]:
assert xgb_search_results.shape[0] == 6
assert set(xgb_search_results.index) == {
    config["candidate"]
    for config in xgb_candidate_configs
}
assert xgb_search_results["ap_mean"].between(0,1).all()
assert xgb_search_results["ap_std"].ge(0).all()

In [ ]:
best_xgb_name = xgb_search_results["ap_mean"].idxmax()
best_xgb_pipeline = xgb_candidate_pipeline[best_xgb_name]
best_xgb_cv_scores = xgb_candidate_cv_scores[best_xgb_name]

best_xgb_summary = xgb_search_results.loc[best_xgb_name]

print("Selected XGBoost candidate:", best_xgb_name)
best_xgb_summary

why not nested CV
- nested CV 能更严格地估计调参后的泛化表现；
- 但它显著增加概念、计算和代码复杂度；
- 本项目只比较六组预先声明的参数，并保留一次 final holdout；
- 因此选择小规模、透明搜索，并如实披露 selection optimism。

In [ ]:
logistic_cv_scores = cross_validate(
    estimator=logistic_pipeline,
    X=X_train,
    y=y_train,
    cv=cv_splits,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
    error_score="raise"
)

In [ ]:
paired_ap_comparison = pd.DataFrame(
    {
        "fold": range(1, len(cv_splits) + 1),
        "logistic_ap": logistic_cv_scores["test_ap"],
        "xgboost_ap": best_xgb_cv_scores["test_ap"],
    }
)

paired_ap_comparison["xgb_minus_logistic"] = (
    paired_ap_comparison["xgboost_ap"]
    - paired_ap_comparison["logistic_ap"]
)

paired_ap_comparison

In [ ]:
paired_ap_summary = pd.Series(
    {
        "mean_ap_difference": paired_ap_comparison[
            "xgb_minus_logistic"
        ].mean(),
        "std_ap_difference": paired_ap_comparison[
            "xgb_minus_logistic"
        ].std(ddof=1),
        "xgb_fold_wins": int(
            (paired_ap_comparison["xgb_minus_logistic"] > 0).sum()
        ),
        "total_folds": len(paired_ap_comparison),
    }
)

In [ ]:
paired_ap_summary

In [ ]:
assert len(paired_ap_comparison) == 5

ap_values = paired_ap_comparison[
    ["logistic_ap", "xgboost_ap"]
]

assert (
    ap_values.ge(0)
    & ap_values.le(1)
).all().all()

assert paired_ap_summary["total_folds"] == 5

In [ ]:
selected_xgb_row = xgb_search_results.loc[[best_xgb_name]].copy()
selected_xgb_row.index = [f"XGBoost ({best_xgb_name})"]

stage4_model_comparison = pd.concat(
    [
        baseline_cv_results,
        selected_xgb_row,
    ]
).sort_values("ap_mean", ascending=False)

stage4_model_comparison


In [ ]:
assert "Dummy" in stage4_model_comparison.index
assert "Logistic Regression" in stage4_model_comparison.index
assert any(
    model_name.startswith("XGBoost")
    for model_name in stage4_model_comparison.index
)
assert stage4_model_comparison.shape[0] == 3

| Component | Data allowed to fit/select it | Protection |
|---|---|---|
| Numeric imputation | Current CV training fold | Inside each Pipeline |
| One-Hot vocabulary | Current CV training fold | Inside each Pipeline |
| Logistic scaling | Current CV training fold | Logistic Pipeline only |
| XGBoost parameters | Training CV results only | Six pre-specified candidates |
| Candidate comparison | Same five training folds | Shared `cv_splits` |
| Threshold | Not selected | Deferred to Stage 5 |
| SHAP explanations | Not produced | Deferred until model selection |
| Final holdout | No model access | Still sealed |

## Stage 4 closeout — 2026-09-05

Fresh execution reproduced B_shallow AP = 0.670135 ± 0.021588 and Logistic Regression AP = 0.661436 ± 0.021602. Paired AP improvement = +0.008699 ± 0.005601; XGBoost won all five folds. The selected candidate for Stage 5 is XGBoost B_shallow (300 trees, learning rate 0.05, depth 2). This is a modest observed ranking improvement, not proof of business impact.

Learning clarification: the draft value 0.67 is the model's AP, not the paired improvement; the formal conclusion above correctly reports +0.0087. The original draft is retained. Fold dependence from overlapping training samples, the small number of folds, and choosing the winner on these same folds all limit statistical claims. Out-of-fold predictions in Stage 5 will protect fold-level fitting boundaries but cannot undo hyperparameter-selection optimism. Earlier full-data EDA exposure must also remain disclosed.

Bank workflow transfer: before comparing models, I would confirm the requesting department's objective, SQL extraction population, customer/account grain, cutoff and field definitions with the data owner. The additional complexity would need documented value under the same validation design and the department's review capacity. This is a hypothetical transfer from my requirements and SQL experience; the Telco model was not trained or deployed at the bank.

Next: training OOF（out-0f-fold） scores, an explicitly hypothetical capacity rule, a frozen threshold decision, and then one final holdout evaluation. No holdout performance has been calculated for this closeout.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import(
    confusion_matrix,precision_score,recall_score,f1_score,
    accuracy_score,average_precision_score,roc_auc_score,
    PrecisionRecallDisplay, ConfusionMatrixDisplay,
)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
assert best_xgb_name == "B_shallow"

In [ ]:
coverage = np.zeros(len(X_train),dtype=int)

In [ ]:
for train_idx, valid_idx in cv_splits:
    assert set(train_idx).isdisjoint(valid_idx)
    coverage[valid_idx] +=1
assert (coverage == 1).all()

In [ ]:
oof_probability= cross_val_predict(
    clone(best_xgb_pipeline), X_train, y_train,
    cv=cv_splits, method = "predict_proba", n_jobs=1,
)[:, 1]

In [ ]:
assert len(oof_probability) == len(y_train)
assert np.isfinite(oof_probability).all()
assert ((oof_probability >= 0) & (oof_probability <= 1)).all()

In [ ]:
def metrics_at_threshold(y_true, probability, threshold):
    predicted = (np.asarray(probability) >= threshold).astype(int)
    tn,fp,fn,tp = confusion_matrix(
        y_true, predicted, labels=[0,1]
    ).ravel()
    return{
        "threshold": float(threshold),
        "recall": recall_score(y_true, predicted, zero_division=0),
        "precision": precision_score(y_true, predicted, zero_division=0),
        "f1": f1_score(y_true, predicted,zero_division=0),
        "accuracy": accuracy_score(y_true, predicted),
        "tn": int(tn), "fp":int(fp), "fn":int(fn), "tp":int(tp),
        "selected_count": int(predicted.sum()),
        "selected_share": float(predicted.mean()),
    }
        

In [ ]:
threshold_table = pd.DataFrame([
    metrics_at_threshold(y_train, oof_probability, t)
    for t in np.round(np.arange(0.05, 1.0, 0.05), 2)
])
threshold_table

In [ ]:
capacity_share = 0.20
capacity_count = int(np.floor(capacity_share*len(y_train)))
levels, counts = np.unique(oof_probability, return_counts=True)
levels, counts = levels[::-1],counts[::-1]
cumulative = np.cumsum(counts)
eligible = np.flatnonzero(cumulative <= capacity_count)
assert len(eligible) > 0, "No nonempty score group fits the capacity."
selected_threshold = float(levels[eligible[-1]])
selected_oof_metrics = metrics_at_threshold(
    y_train, oof_probability, selected_threshold
)
assert 0 < selected_oof_metrics["selected_count"] <= capacity_count
pd.Series(selected_oof_metrics)

I selected the operating threshold using out-of-fold predictions from the training data under a hypothetical outreach capacity of 20%.

The selected threshold was approximately 0.5171. It flagged 1,126 of the 5,634 training customers, representing 19.99% of the training population.

At this threshold, the model identified 758 of the 1,495 customers who actually churned, producing a recall of 50.70%. Among the 1,126 customers selected for outreach, 758 actually churned, resulting in a precision of 67.32%.

The decision produced 368 false positives. These customers would consume outreach capacity even though they did not churn. It also produced 737 false negatives, meaning that these customers actually churned but were not included in the candidate list.

I accepted this trade-off because the hypothetical 20% capacity prevents the business from contacting every customer who may be at risk. The selected rule prioritizes higher-risk customers without exceeding that capacity, and approximately two-thirds of the selected customers were actual churners. However, it still missed almost half of the churned customers, so I would present this as a capacity-constrained trade-off rather than a proven optimal business policy.

The threshold was selected entirely from training-data out-of-fold predictions. After the final model is refitted on the complete training set, its probability scale may change. Future customer populations may also have different score distributions. Therefore, a fixed threshold of 0.5171 is not guaranteed to select exactly 20% of future customers.

If the operational requirement is to contact exactly the highest-risk 20% in every batch, I would define a top-k ranking and tie-breaking rule rather than retuning the threshold on the final holdout. A fixed threshold controls the minimum risk score, while a top-k rule controls the number of customers selected.

In [ ]:
decision_record = {
    "model": best_xgb_name,
    "threshold": selected_threshold,
    "capacity_share_assumption":capacity_share,
    "seed": RANDOM_STATE,
    "selection_source": "training OOF scores; holdout not inspected",
    "xgb_parameters": best_xgb_pipeline.named_steps["model"].get_params(),
}
decision_record

In [ ]:
final_pipeline = clone(best_xgb_pipeline).fit(X_train,y_train)
holdout_probability = final_pipeline.predict_proba(X_holdout)[:,1]
holdout_metrics = metrics_at_threshold(
    y_holdout, holdout_probability, selected_threshold
)
holdout_metrics.update({
    "average_precision": average_precision_score(y_holdout,holdout_probability),
    "roc_auc": roc_auc_score(y_holdout, holdout_probability),
})

In [ ]:
pd.Series(holdout_metrics)

In [ ]:
holdout_prediction = (holdout_probability >= selected_threshold).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_holdout, holdout_prediction, labels=[0, 1],
    display_labels=["No churn", "Churn"],
)
plt.show()
PrecisionRecallDisplay.from_predictions(y_holdout, holdout_probability)
plt.show()
RocCurveDisplay.from_predictions(y_holdout, holdout_probability)
plt.show()

In [ ]:
assert X_holdout.index.equals(id_holdout.index)
prediction_sample = pd.DataFrame({
    "customerID": id_holdout.to_numpy(),
    "churn_probability": holdout_probability,
    "predicted_class": holdout_prediction,
    "risk_tier": np.where(holdout_prediction == 1, "High", "Standard"),
})
assert prediction_sample["customerID"].is_unique
assert len(prediction_sample) == len(X_holdout)
prediction_sample.head(10)

SHAP：
SHAP值 > 0
→ 该特征把预测推向更高流失风险

SHAP值 < 0
→ 该特征把预测推向更低流失风险

In [ ]:
import shap

In [ ]:
from xgboost import DMatrix

In [ ]:
explanation_customers = X_train.sample(n=200, random_state=RANDOM_STATE)

In [ ]:
transformed = final_pipeline.named_steps["preprocess"].transform(explanation_customers)

In [ ]:
import shap
from xgboost import DMatrix

explanation_customers = X_train.sample(
    n=200,
    random_state=RANDOM_STATE,
)

transformed = final_pipeline.named_steps[
    "preprocess"
].transform(explanation_customers)

if hasattr(transformed, "toarray"):
    transformed = transformed.toarray()

feature_names = final_pipeline.named_steps[
    "preprocess"
].get_feature_names_out()

booster = final_pipeline.named_steps[
    "model"
].get_booster()

matrix = DMatrix(transformed)

contributions = booster.predict(
    matrix,
    pred_contribs=True,
)

raw_margin = booster.predict(
    matrix,
    output_margin=True,
)

assert np.allclose(
    contributions.sum(axis=1),
    raw_margin,
    atol=1e-5,
)

explanation = shap.Explanation(
    values=contributions[:, :-1],
    base_values=contributions[:, -1],
    data=transformed,
    feature_names=list(feature_names),
)

assert explanation.values.shape == transformed.shape

shap.plots.beeswarm(
    explanation,
    max_display=15,
)

local_index = 0  # Predetermined first sampled customer, not cherry-picked.

shap.plots.waterfall(
    explanation[local_index],
    max_display=15,
)

1. OOF predictions are more reliable than in-sample predictions because each customer is scored by a model that was trained without that customer. This provides a more realistic basis for threshold selection. However, OOF predictions do not remove all selection bias. The XGBoost candidate was selected from six configurations using the same folds, and the earlier full-data EDA created analyst-contamination risk.

2. The 20% outreach capacity is a hypothetical assumption, not a verified operational fact. The current threshold rule includes complete tied-score groups without exceeding capacity. If the business requires an exact number of customers, I would define a top-k rule and a predetermined tie-breaking policy.

3. At the selected OOF threshold of 0.5171, the model identified 758 actual churners and missed 737, producing a recall of 50.70%. On the final holdout, the frozen rule identified 185 churners and missed 189, producing a recall of 49.47%.

4. False positives consume outreach, review, communication, and potentially incentive resources. False negatives are actual churners who were not selected, but they cannot automatically be treated as recoverable losses. The dataset does not show whether an intervention would have changed their outcome or how much value could have been retained.

5. The B_shallow model and the 0.5171 threshold were frozen using training cross-validation and OOF predictions before the holdout was inspected. No model or threshold decision should be changed based on the final holdout results.

6. A fixed threshold controls the minimum risk score rather than the number of customers selected. Model refitting and future changes in the customer and score distributions may cause the selected share to differ from 20%. A strict capacity requirement should therefore use a top-k or queueing rule.

7. In the current implementation, SHAP values are expressed on the XGBoost raw-margin scale, which is generally log-odds for binary classification. They explain how features contribute to the model output, but they do not establish causality because the model is based on observational associations and may be affected by confounding and correlated features.

8. For a hypothetical bank work order, I would confirm the business owner responsible for reviewing the candidate list, the operational outreach capacity, the required customer or account-level output grain, and the approved fields and access permissions. I would also confirm masking, audit, and intervention-feedback requirements with the relevant business, system, and governance roles.